# Estrutura tabular, limpeza e qualidade

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_03/02_estrutura_limpeza_e_qualidade.ipynb)

O Notebook 01 produziu representações legíveis e um inventário de riscos.
Agora precisamos organizar essas representações segundo unidades explícitas e
aplicar correções sem apagar os valores recebidos ou suas incertezas.

## 1. Uma variável por coluna, uma observação por linha

A estrutura tabular depende da unidade de análise. Valores atômicos e nomes de
campos estáveis facilitam seleção e junção, mas não determinam sozinhos a
ontologia correta. Tabelas separadas podem representar documentos, pessoas,
lugares e relações sem repetir tudo em uma única linha.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import pandas as pd
bruto = pd.read_csv("dados/brutos/catalogo_messy.csv", sep=";", dtype={"codigo_municipio": "string"})
bruto.info()

A inspeção da tabela mostra que colunas e linhas já carregam decisões sobre
entidades e observações. A comparação entre formatos largo e longo torna
visível como uma reorganização pode mudar precisamente a unidade da linha.

## 2. Largo e longo

No formato largo, período e tema aparecem nos nomes das colunas. No longo,
essas dimensões viram valores. A transformação altera a unidade da linha: de
documento para combinação documento–tema–período.

![Uma tabela larga com uma linha por documento é reorganizada em tabela longa com uma linha por combinação documento, tema e período, sem resumir os valores.](imagens/02_largo_longo.svg)

Observe que `melt` não apenas muda a aparência: ele redefine o que cada linha
representa. Essa unidade precisa constar no dicionário de dados.

In [ ]:
largo = pd.read_csv("dados/brutos/indicadores_largos.csv")
longo = largo.melt(id_vars="id_documento", var_name="tema_periodo", value_name="ocorrencias")
partes = longo["tema_periodo"].str.extract(r"(?P<tema>.+)_(?P<periodo>\d{4})")
longo = pd.concat([longo[["id_documento", "ocorrencias"]], partes], axis=1)
print("Largo:", largo.shape, "| Longo:", longo.shape)
longo.head(6)

A transformação para o formato longo reorganiza valores sem precisar apagar a
entrada. O mesmo princípio vale para correções textuais: em vez de substituir
silenciosamente, criaremos uma representação normalizada ao lado do original.

## 3. Preservar original, criar versão normalizada

Não sobrescreveremos títulos, municípios ou gêneros. A coluna original permite
auditar o mapa de equivalências e recuperar distinções apagadas por uma regra.
A remoção de acentos pode ajudar correspondência aproximada, mas não deve
substituir automaticamente a grafia de apresentação.

In [ ]:
import re
import unicodedata

def chave_textual(valor):
    if pd.isna(valor):
        return pd.NA
    texto = " ".join(str(valor).strip().lower().split())
    texto = "".join(c for c in unicodedata.normalize("NFD", texto) if unicodedata.category(c) != "Mn")
    return texto

trabalho = bruto.copy()
trabalho["titulo_chave"] = trabalho["titulo"].map(chave_textual)
trabalho["municipio_chave"] = trabalho["municipio"].map(chave_textual)
mapa_generos = {"editorial": "editorial", "notícia": "notícia", "noticia": "notícia", "carta": "carta", "manifesto": "manifesto"}
trabalho["genero_original"] = trabalho["genero"]
trabalho["genero_padronizado"] = trabalho["genero"].map(chave_textual).map(mapa_generos)
trabalho[["titulo", "titulo_chave", "genero_original", "genero_padronizado"]]

As chaves normalizadas facilitam comparações, mas cada tipo de campo exige uma
regra própria. Datas incompletas, identificadores numéricos e valores ausentes
não podem ser tratados como simples variações ortográficas.

## 4. Datas, códigos e ausências

Código de município é identificador textual, não quantidade. Datas parciais
não devem receber dia e mês inventados. Um parser explícito separa datas
completas, anos isolados e valores desconhecidos; guardar o original, a
precisão e a razão evita transformar incerteza em falsa exatidão.

![O valor de data recebido é preservado, passa por uma regra explícita e produz representação derivada, precisão e razão; log e testes permitem retornar à fonte.](imagens/02_transformacao_rastreavel.svg)

No exemplo, uma data completa pode ocupar `data_normalizada`; um ano isolado
ocupa `ano_documento`, mas não recebe dia e mês arbitrários. A coluna
`precisao_data` distingue `dia`, `ano`, `desconhecida` e, se ocorrer, `inválida`.

In [ ]:
trabalho["data_original"] = trabalho["data_documento"]

def decompor_data(valor):
    texto = str(valor).strip()
    formatos = [
        (r"\d{4}-\d{2}-\d{2}", "%Y-%m-%d"),
        (r"\d{2}/\d{2}/\d{4}", "%d/%m/%Y"),
    ]
    for padrao, formato in formatos:
        if re.fullmatch(padrao, texto):
            data = pd.to_datetime(texto, format=formato, errors="coerce")
            if pd.notna(data):
                return data, data.year, "dia", pd.NA
            return pd.NaT, pd.NA, "inválida", "data impossível no calendário"
    if re.fullmatch(r"\d{4}", texto):
        return pd.NaT, int(texto), "ano", "dia e mês não informados"
    if texto.casefold() == "data desconhecida":
        return pd.NaT, pd.NA, "desconhecida", "data não informada"
    return pd.NaT, pd.NA, "inválida", "formato não reconhecido"

datas = trabalho["data_original"].apply(decompor_data).apply(pd.Series)
datas.columns = ["data_normalizada", "ano_documento", "precisao_data", "razao_data_ausente"]
datas["ano_documento"] = datas["ano_documento"].astype("Int64")
trabalho[datas.columns] = datas
trabalho["palavras"] = pd.to_numeric(trabalho["palavras"], errors="coerce")
trabalho["razao_palavras_ausente"] = trabalho["palavras"].isna().map({True: "não contado", False: pd.NA})
trabalho[["data_original", "data_normalizada", "ano_documento", "precisao_data", "razao_data_ausente", "palavras"]]

Depois de normalizar campos e representar ausências, podemos comparar registros
com maior consistência. Ainda assim, semelhança não prova identidade: ela apenas
produz candidatos a duplicata que precisam de revisão documental.

## 5. Duplicatas são uma hipótese

IDs repetidos detectam um tipo de duplicata. Registros de um mesmo documento
com IDs diferentes exigem combinação de campos e revisão. Remover pelo título
isolado poderia apagar edições legítimas.

In [ ]:
trabalho["possivel_duplicata"] = trabalho.duplicated(
    subset=["titulo_chave", "data_normalizada", "municipio_chave", "palavras"],
    keep=False,
)
trabalho.loc[trabalho["possivel_duplicata"], ["id_documento", "titulo", "data_original", "palavras"]]

As regras anteriores modificaram representações, registraram incertezas e
sinalizaram casos para revisão. Antes de usar essa tabela em junções, precisamos
resumir o que mudou e exportar uma camada intermediária reproduzível.

## 6. Relatório e exportação intermediária

Antes/depois deve quantificar transformações, falhas e casos para revisão. A
saída intermediária não substitui os dados brutos.

In [ ]:
relatorio = {
    "linhas": len(trabalho),
    "datas_sem_precisao_de_dia": int(trabalho["data_normalizada"].isna().sum()),
    "datas_parciais_ano": int(trabalho["precisao_data"].eq("ano").sum()),
    "datas_desconhecidas": int(trabalho["precisao_data"].eq("desconhecida").sum()),
    "datas_invalidas": int(trabalho["precisao_data"].eq("inválida").sum()),
    "palavras_ausentes": int(trabalho["palavras"].isna().sum()),
    "generos_sem_mapeamento": int(trabalho["genero_padronizado"].isna().sum()),
    "registros_em_grupos_de_possiveis_duplicatas": int(trabalho["possivel_duplicata"].sum()),
}
trabalho.to_csv("dados/intermediarios/catalogo_normalizado.csv", index=False)
pd.Series(relatorio, name="quantidade")

O relatório quantitativo informa quantos casos foram afetados, mas não explica
por que cada decisão foi tomada. O log acrescenta regra, justificativa, teste,
reversibilidade e responsabilidade a esse resumo.

## Atividade — log de transformação

Registre campo, problema, regra, justificativa, valores afetados, teste,
reversibilidade e responsável. Explique que distinção cada regra pode apagar.
**Log:** Escreva aqui.

O log da atividade liga cada valor derivado ao original e à regra aplicada. A
síntese recupera essa relação antes de a tabela intermediária participar de
junções com outras entidades.

## Síntese

Limpeza responsável acrescenta rastreabilidade. Ela não transforma incerteza
substantiva em certeza técnica nem autoriza exclusão silenciosa. No Notebook
03, a tabela intermediária e seu log serão usados para declarar chaves,
cardinalidades e vínculos entre documentos, textos, temas e indicadores.